In [1]:
#utils
import pandas as pd
import numpy as np
import contextlib
import os
import warnings
from scipy.stats import norm
import sys
import time 

#visualization libraries
import matplotlib.pyplot as plt

#machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve

#models 
from models.AutoEncoder import Autoencoder
from models.ContractiveAutoEncoder import ContractiveAutoEncoder
from models.VariationalAutoencoder import VariationalAutoEncoder
from models.alad.ALAD import ALAD


warnings.filterwarnings("ignore")

d:\programs\conda\envs\macchina\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
df = pd.read_csv('SimulationData.csv')
df = df.head(1000)

In [3]:
target_feature = "isghost"
columns = df.drop(columns=target_feature).columns
#instantiate two more datasets one to contain the normal data
#and the other to hold the anomalous instances
train_normal_data = df[df[target_feature] == 0].drop(columns=target_feature)
anomalous_data = df[df[target_feature] == 1].drop(columns=target_feature)

In [4]:
def get_normal_data(df):
    df = df.sample(frac=1)
    features = df.drop([target_feature], axis=1)
    label = df[target_feature]

    #the stratify parameter here is set to be "isghost" meaning that the 
    #proportions of ghost and normal tracks in both train and test datasets
    #will be kept the same...
    x_train, x_test, y_train, y_test = train_test_split(
        features, label, test_size=0.2, stratify=label
    )

    #remember that at least for the autoencoders we're gonna use just the 
    #normal instances during the training phase so we have to filter those in x_train
    train_index = y_train[y_train == 0].index
    train_normal_data = x_train.loc[train_index]

    return train_normal_data, x_train, y_train, x_test, y_test

In [5]:
def standardize_normalize(train_x, normal_train_data, test_data):


    non_zero_cols = train_x.columns[(train_x != 0).any()]

    # Calculate mean and standard deviation only for non-zero columns
    mean = train_x[non_zero_cols].mean()
    std = train_x[non_zero_cols].std() + sys.float_info.min

    # Standardize the non-zero columns in normal_train_data, train_x, and test_data
    normal_train_data[non_zero_cols] = (normal_train_data[non_zero_cols] - mean) / std
    train_x[non_zero_cols] = (train_x[non_zero_cols] - mean) / std
    test_data[non_zero_cols] = (test_data[non_zero_cols] - mean) / std

    # Create a scaler and apply Min-Max scaling to all columns
    scaler = MinMaxScaler()
    normal_train_data = scaler.fit_transform(normal_train_data)
    train_x = scaler.transform(train_x)
    test_data = scaler.transform(test_data)
    
    return train_x, normal_train_data, test_data

In [6]:
train_normal_data, x_train, y_train, x_test, y_test = get_normal_data(df)
x_train, train_normal_data, x_test = standardize_normalize(x_train, train_normal_data, x_test)

In [7]:
INPUT_DIM = train_normal_data.shape[1]
ENCODING_DIM = 3
learing_rates = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05]

In [8]:
def get_data():

    train_normal_data, x_train, _, x_test, y_test = get_normal_data(df.sample(frac=1).reset_index(drop=True))
    _, train_normal_data, x_test = standardize_normalize(x_train, train_normal_data, x_test)

    return train_normal_data, x_test, y_test

In [9]:
def build_models(learning_rate, encoding_dim):

    ae = Autoencoder(INPUT_DIM, encoding_dim, learning_rate)
    cae = ContractiveAutoEncoder(INPUT_DIM, encoding_dim, lambda_=0.01, learning_rate = learning_rate)
    vae = VariationalAutoEncoder(INPUT_DIM, encoding_dim, learning_rate)
    alad = ALAD(input_dim=INPUT_DIM, latent_dim=encoding_dim, learning_rate=learning_rate)

    ae.build(input_shape=(train_normal_data.shape))
    cae.build(input_shape=(train_normal_data.shape))
    vae.build(input_shape=(train_normal_data.shape))
    
    return ae, cae, vae, alad

In [10]:
def z_test(values1, values2, name1, name2, title):

    critical = 1.645
    averages_vae = dict(list(values1[0].items())[-5:])
    means = pd.DataFrame([float(value) for value in list(averages_vae.values())], columns=["Avg " + name1])
    std = pd.DataFrame([float(value) for value in list(values1[2].values())], columns=["Sigma " + name1])

    averages_vae_no_transfer = dict(list(values2[0].items())[-5:])
    means_no = pd.DataFrame([float(value) for value in list(averages_vae_no_transfer.values())], columns=["Avg " + name2])
    std_no = pd.DataFrame([float(value) for value in list(values2[2].values())], columns=["Sigma " + name2])

    #n is equal to the iterations of the mean_confidence method
    n = 5
    numer = means['Avg ' + name1] - means_no['Avg ' + name2]
    denom = np.sqrt((std['Sigma ' + name1]**2 / n) + (std_no['Sigma ' + name2]**2 / n))
    z_score = numer / denom
    p_value = 2 * (1 - norm.cdf(np.abs(z_score)))

    df = pd.DataFrame({'z-score': z_score, 'p-value': p_value})
    df['Decision'] = ['Reject Null' if z > critical else "Don't reject null" for z in df['z-score']]
    df.index = ["Accuracy", "Recall", "Precision", "F1", "Training time", "AUC"]
    df.name = title
    print(df.head())

In [11]:

def avgNestedLists(nested_vals):
    """
    Averages a 2-D array and returns a 1-D array of all of the columns
    averaged together, regardless of their dimensions.
    """
    output = []
    maximum = 0
    for lst in nested_vals:
        if len(lst) > maximum:
            maximum = len(lst)
    for index in range(maximum): # Go through each index of longest list
        temp = []
        for lst in nested_vals: # Go through each list
            if index < len(lst): # If not an index error
                temp.append(lst[index])
        output.append(np.nanmean(temp))
    return output

In [12]:
import numpy as np
from sklearn.metrics import roc_curve, auc
from scipy import stats


def mean_and_confidence(model, threshold_function, alpha=0.95, iterations=30, transfer=False, epochs=5, learning_rate = 0.003):
   

    #initialize all the containers for teh metrics to track
    list_fpr = []
    list_tpr = []
    list_auc = []

    #precision and recall will be used for 
    #the curve but also as general parameters 
    list_precision_curve = []
    list_recall_curve = []

    #classification metrics 
    list_accuracy = []
    list_precision = []
    list_recall = []
    list_f1 = []

    list_training_time = []

    model.save_weights('saved_weights/exp2/my_model' +  model.model_name)

    

    # Perform 100 iterations
    for _ in range(iterations):

        x_train, x_test, y_test = get_data()

        history, training_time = model.train(x_train, epochs=epochs, batch_size=30, validation_split=0.2)
        ###time each model 
        threshold = threshold_function(x_train, x_test, y_test)
        predictions = model.find_anomalies(x_test, threshold)
        accuracy, precision, recall, f1 = model.classification_metrics(y_test, predictions)

        #re-set all the trainable variables of the model
        model.load_weights('saved_weights/exp2/my_model' +  model.model_name)
        
        fpr_ae, tpr_ae, _ = roc_curve(y_test, predictions)
        auc_ae = auc(fpr_ae, tpr_ae)
        list_fpr.append(fpr_ae)
        list_tpr.append(tpr_ae)
        

        precision_curve, recall_curve, _ = precision_recall_curve(y_test, predictions)
        list_precision_curve.append(precision_curve)
        list_recall_curve.append(recall_curve)


        list_accuracy.append(accuracy)
        list_precision.append(precision)
        list_recall.append(recall)
        list_f1.append(f1)
        list_training_time.append(training_time)
        list_auc.append(auc_ae)

    #roc_curve can return different sized arrays
    #this means that ther needs to be a custom function
    #to compute the mean 
    avg_fpr = avgNestedLists(list_fpr)
    avg_tpr = avgNestedLists(list_tpr)
    avg_precision_curve = avgNestedLists(list_precision_curve)
    avg_recall_curve = avgNestedLists(list_recall_curve)
    avg_acc = np.mean(list_accuracy)
    avg_rec = np.mean(list_recall)
    avg_pre = np.mean(list_precision)
    avg_f1 = np.mean(list_f1)
    avg_time = np.mean(list_training_time)
    avg_auc = np.mean(list_auc)


    #standard deviations
    accuracy_std = np.std(list_accuracy)
    precision_std = np.std(list_precision)
    recall_std = np.std(list_recall)
    f1_std = np.std(list_f1)
    training_time_std = np.std(list_training_time)
    auc_std = np.std(list_auc)

    #how to take the "mean curve"  
    interval_acc = stats.t.interval(alpha=alpha, loc=np.mean(list_accuracy), df=len(list_accuracy)-1, scale=stats.sem(list_accuracy))
    interval_rec = stats.t.interval(alpha=alpha, loc=np.mean(list_recall), df=len(list_recall)-1, scale=stats.sem(list_recall))
    interval_pre = stats.t.interval(alpha=alpha, loc=np.mean(list_precision), df=len(list_precision)-1, scale=stats.sem(list_precision))
    interval_f1 = stats.t.interval(alpha=alpha, loc=np.mean(list_f1), df=len(list_f1)-1, scale=stats.sem(list_f1))
    interval_time = stats.t.interval(alpha=alpha, loc=np.mean(list_training_time), df=len(list_training_time)-1, scale=stats.sem(list_training_time))
    interval_auc = stats.t.interval(alpha=alpha, loc=np.mean(list_auc), df=len(list_auc)-1, scale=stats.sem(list_auc))




    averages = {
        "Average FPR": avg_fpr,
        "Average TPR": avg_tpr,
        "Average precision curve": avg_precision_curve,
        "Average recall curve": avg_recall_curve,
        "Average accuracy": avg_acc,
        "Average recall": avg_rec,
        "Average precision": avg_pre,
        "Average F-1": avg_f1,
        "Average training time": avg_time,
        "Average AUC": avg_auc,
        }
    

    sigmas = {
        "STD accuracy": accuracy_std,
        "STD recall": recall_std,
        "STD precision": precision_std,
        "STD F-1": f1_std,
        "STD training time": training_time_std,
        "STD AUC": auc_std,
    }

    
    intervals = {
        "Interval accuracy": interval_acc,
        "Interval recall": interval_rec,
        "Interval precision": interval_pre,
        "Interval F-1": interval_f1,
        "Interval training time": interval_time,
        "Interval AUC": interval_auc,
        }
    
    return [averages, intervals, sigmas]

# Experiment with learning rate

In [13]:
with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    alad_list = []
    ae_list = []
    cae_list = []
    vae_list = []

    for rate in learing_rates:

        ae, cae, vae, alad = build_models(encoding_dim=ENCODING_DIM, learning_rate=rate)
        
        t = mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)
        alad_list.append(t[0])
        t = mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)
        ae_list.append(t[0])
        t = mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)
        cae_list.append(t[0])
        t = mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)
        vae_list.append(t[0])

### I'm using the recall as a parameter ot be optimized for the learning_rate search.

In [14]:
from collections import Counter


def find_best(method_list, variable_list):

    maxx = 0
    best_rate = variable_list[0]

    for index, value in enumerate(method_list):

        if value["Average recall"] > maxx:
            maxx = value["Average recall"]
            best_rate = variable_list[index]

    return str(best_rate)

In [15]:
rates_found = [find_best(alad_list, learing_rates), find_best(ae_list, learing_rates), find_best(cae_list, learing_rates), find_best(vae_list, learing_rates)]
print("the best rate for alad method is " + find_best(alad_list, learing_rates))
print("the best rate for ae method is " + find_best(ae_list, learing_rates))
print("the best rate for cae method is " + find_best(cae_list, learing_rates))
print("the best rate for vae method is " + find_best(vae_list, learing_rates))

occurence_count = Counter(rates_found)
LEARNING_RATE = occurence_count.most_common(1)[0][0]
print("this is the final learning rate " + str(LEARNING_RATE))

the best rate for alad method is 0.003
the best rate for ae method is 0.05
the best rate for cae method is 0.001
the best rate for vae method is 0.01
this is the final learning rate 0.003


# Experiment with encoding dimension

In [16]:
encoding_dims = list(range(2, round(INPUT_DIM / 2),  1))
encoding_dims = [2, 3, 5]

with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    alad_list = []
    ae_list = []
    cae_list = []
    vae_list = []

    for encoding_dim in encoding_dims:

        ae, cae, vae, alad = build_models(encoding_dim=encoding_dim, learning_rate=0.001)
        
        t = mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)
        alad_list.append(t[0])
        t = mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)
        ae_list.append(t[0])
        t = mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)
        cae_list.append(t[0])
        t = mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)
        vae_list.append(t[0])

In [17]:
enc_found = [find_best(alad_list, encoding_dims), find_best(ae_list, encoding_dims), find_best(cae_list, encoding_dims), find_best(vae_list, encoding_dims)]
print("the best encoding_dim for alad method is " + find_best(alad_list, encoding_dims))
print("the best encoding_dim for ae method is " + find_best(ae_list, encoding_dims))
print("the best encoding_dim for cae method is " + find_best(cae_list, encoding_dims))
print("the best encoding_dim for vae method is " + find_best(vae_list, encoding_dims))


occurence_count = Counter(enc_found)
ENCODING_DIM = int(occurence_count.most_common(1)[0][0])
print("this is the final encoding dim " + str(ENCODING_DIM))

the best encoding_dim for alad method is 3
the best encoding_dim for ae method is 2
the best encoding_dim for cae method is 2
the best encoding_dim for vae method is 5
this is the final encoding dim 2


# Experiment with other methods

In [18]:
from sklearn import svm

def SVM_predict(train_normal_data, x_test):
    
    clf = svm.OneClassSVM(nu=0.1, kernel="rbf", gamma=0.2)
    clf.fit(train_normal_data)
    # predict if the test instaces are anomalies or not
    #Returns -1 for outliers and 1 for inliers.
    y_pred = clf.predict(x_test)  
    y_pred = [1 if pred == -1 else 0 for pred in y_pred]
    
    return y_pred


In [19]:
from sklearn.ensemble import IsolationForest

def IF_predict(x_train, x_test):

    clf = IsolationForest(n_estimators=10000, max_samples='auto', contamination=0.2, random_state=42)
    clf.fit(x_train)
    y_pred = clf.predict(x_test)
    y_pred = [1 if pred == -1 else 0 for pred in y_pred]

    return y_pred


In [20]:
from sklearn.cluster import DBSCAN

def DBSCAN_predict(x_train, x_test):

    dbscan = DBSCAN(eps=0.3, min_samples=3)
    dbscan.fit(x_train)
    labels = dbscan.fit_predict(x_test)
    y_pred = [1 if label != -1 else 0 for label in labels]
    
    return y_pred

In [21]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score


def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    return accuracy, precision, recall, f1

In [22]:
def test_methods(function, alpha=0.95, iterations=30):
    #initialize all the containers for teh metrics to track
    list_fpr = []
    list_tpr = []
    list_auc = []

    #precision and recall will be used for 
    #the curve but also as general parameters 
    list_precision_curve = []
    list_recall_curve = []

    #classification metrics 
    list_accuracy = []
    list_precision = []
    list_recall = []
    list_f1 = []


    for _ in range(iterations):

        x_train, x_test, y_test = get_data()

        predictions = function(x_train, x_test)
        
        accuracy, precision, recall, f1 = calculate_metrics(y_test, predictions)

        fpr_ae, tpr_ae, _ = roc_curve(y_test, predictions)
        auc_ae = auc(fpr_ae, tpr_ae)
        list_fpr.append(fpr_ae)
        list_tpr.append(tpr_ae)
        

        precision_curve, recall_curve, _ = precision_recall_curve(y_test, predictions)
        list_precision_curve.append(precision_curve)
        list_recall_curve.append(recall_curve)


        list_accuracy.append(accuracy)
        list_precision.append(precision)
        list_recall.append(recall)
        list_f1.append(f1)
        list_auc.append(auc_ae)

    #roc_curve can return different sized arrays
    #this means that ther needs to be a custom function
    #to compute the mean 
    avg_fpr = avgNestedLists(list_fpr)
    avg_tpr = avgNestedLists(list_tpr)
    avg_precision_curve = avgNestedLists(list_precision_curve)
    avg_recall_curve = avgNestedLists(list_recall_curve)
    avg_acc = np.mean(list_accuracy)
    avg_rec = np.mean(list_recall)
    avg_pre = np.mean(list_precision)
    avg_f1 = np.mean(list_f1)
    avg_auc = np.mean(list_auc)


    #standard deviations
    accuracy_std = np.std(list_accuracy)
    precision_std = np.std(list_precision)
    recall_std = np.std(list_recall)
    f1_std = np.std(list_f1)
    auc_std = np.std(list_auc)

    #intervals
    interval_acc = stats.t.interval(alpha=alpha, loc=np.mean(list_accuracy), df=len(list_accuracy)-1, scale=stats.sem(list_accuracy))
    interval_rec = stats.t.interval(alpha=alpha, loc=np.mean(list_recall), df=len(list_recall)-1, scale=stats.sem(list_recall))
    interval_pre = stats.t.interval(alpha=alpha, loc=np.mean(list_precision), df=len(list_precision)-1, scale=stats.sem(list_precision))
    interval_f1 = stats.t.interval(alpha=alpha, loc=np.mean(list_f1), df=len(list_f1)-1, scale=stats.sem(list_f1))
    interval_auc = stats.t.interval(alpha=alpha, loc=np.mean(list_auc), df=len(list_auc)-1, scale=stats.sem(list_auc))




    averages = {
        "Average FPR": avg_fpr,
        "Average TPR": avg_tpr,
        "Average precision curve": avg_precision_curve,
        "Average recall curve": avg_recall_curve,
        "Average accuracy": avg_acc,
        "Average recall": avg_rec,
        "Average precision": avg_pre,
        "Average F-1": avg_f1,
        "Average AUC": avg_auc,
        }
    

    sigmas = {
        "STD accuracy": accuracy_std,
        "STD recall": recall_std,
        "STD precision": precision_std,
        "STD F-1": f1_std,
        "STD AUC": auc_std,
    }

    
    intervals = {
        "Interval accuracy": interval_acc,
        "Interval recall": interval_rec,
        "Interval precision": interval_pre,
        "Interval F-1": interval_f1,
        "Interval AUC": interval_auc,
        }
    
    return [averages, intervals, sigmas]



In [23]:
with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):
    ae, cae, vae, alad = build_models(encoding_dim=ENCODING_DIM, learning_rate=0.001)
        
    alad_std = mean_and_confidence(alad, alad.find_best_deviations_threshold, alpha=0.95)
    ae_std = mean_and_confidence(ae, ae.find_best_deviations_threshold, alpha=0.95)
    cae_std = mean_and_confidence(cae, cae.find_best_deviations_threshold, alpha=0.95)
    vae_std = mean_and_confidence(vae, vae.find_best_deviations_threshold, alpha=0.95)

    if_vals = test_methods(IF_predict, iterations =30)
    svm_vals = test_methods(SVM_predict, iterations =30)
    dbscan_vals = test_methods(DBSCAN_predict, iterations =30)

In [24]:
print("______________AE___________________")
z_test(ae_std, if_vals, "AE", "IF", "AE-IF")
z_test(ae_std, svm_vals,  "AE", "SVM", "AE-SVM")
z_test(ae_std, dbscan_vals, "AE", "DBSCAN", "AE-DBSCAN")
print("______________CAE___________________")
z_test(cae_std, if_vals, "CAE", "IF", "CAE-IF")
z_test(cae_std, svm_vals, "CAE", "SVM", "CAE-SVM")
z_test(cae_std, dbscan_vals, "CAE", "DBSCAN", "CAE-DBSCAN")
print("______________VAE___________________")
z_test(vae_std, if_vals, "VAE", "IF", "VAE-IF")
z_test(vae_std, svm_vals, "VAE", "SVM", "VAE-SVM")
z_test(vae_std, dbscan_vals, "VAE", "DBSCAN", "VAE-DBSCAN")
print("______________ALAD___________________")
z_test(alad_std, if_vals, "ALAD", "IF", "ALAD-IF")
z_test(alad_std, svm_vals, "ALAD", "SVM","ALAD-SVM")
z_test(alad_std, dbscan_vals,"ALAD", "DBSCAN", "ALAD-DBSCAN")

______________AE___________________
                 z-score   p-value           Decision
Accuracy      -17.170830  0.000000  Don't reject null
Recall         -3.453580  0.000553  Don't reject null
Precision       1.104685  0.269296  Don't reject null
F1             79.900905  0.000000        Reject Null
Training time  -0.454610  0.649390  Don't reject null
                 z-score   p-value           Decision
Accuracy      -21.731738  0.000000  Don't reject null
Recall         -0.416817  0.676812  Don't reject null
Precision       1.404300  0.160229  Don't reject null
F1             70.884118  0.000000        Reject Null
Training time   0.218215  0.827261  Don't reject null
                 z-score       p-value           Decision
Accuracy        2.338796  1.934601e-02        Reject Null
Recall         -6.661653  2.707656e-11  Don't reject null
Precision       2.010959  4.432977e-02        Reject Null
F1             90.383601  0.000000e+00        Reject Null
Training time   0.952398  

# Experiment inference time

In [25]:
#return average time alongside confidence interval

def inference_time(model, threshold_function, iterations=2):
   

    x_train, x_test, y_test = get_data()
    
    
    history, training_time = model.train(x_train, epochs=2, batch_size=30, validation_split=0.2)
    threshold = threshold_function(x_train, x_test, y_test)
    
    list_time = []

    for _ in range(iterations):

        start = time.time()
        
        predictions = model.find_anomalies(x_test, threshold)
        finish = time.time()
        
        
        list_time.append(finish-start)
        
        
    interval_time = stats.t.interval(alpha=0.95, loc=np.mean(list_time), df=len(list_time)-1, scale=stats.sem(list_time))
    avg_time = str(np.mean(list_time))
    
    return [avg_time, interval_time, len(x_test)]
        

In [26]:
LEARNING_RATE = float(LEARNING_RATE)

with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull):

    ae, cae, vae, alad = build_models(encoding_dim=ENCODING_DIM, learning_rate=LEARNING_RATE)
     
    alad_time = inference_time(alad, alad.find_best_deviations_threshold)    
    ae_time = inference_time(ae, ae.find_best_deviations_threshold)
    cae_time = inference_time(cae, cae.find_best_deviations_threshold)
    vae_time = inference_time(vae, vae.find_best_deviations_threshold)
    
    
    model_names = ['ALAD', 'AE', 'CAE', 'VAE']
    
    avg_times = []
    avg_times.append(alad_time[0])
    avg_times.append(ae_time[0]) 
    avg_times.append(cae_time[0]) 
    avg_times.append(vae_time[0]) 
    
    instances = []
    instances.append(alad_time[2])
    instances.append(ae_time[2]) 
    instances.append(cae_time[2]) 
    instances.append(vae_time[2]) 
    
    
    interval_times = []
    interval_times.append(alad_time[1])
    interval_times.append(ae_time[1])
    interval_times.append(cae_time[1])
    interval_times.append(vae_time[1])

    data = {'Model': model_names,
        'Average Time': avg_times,
        'Interval': interval_times,
        'Instances': instances,}

In [27]:
inference = pd.DataFrame(data)
inference

,Model,Average Time,Interval,Instances
0,ALAD,0.03499901294708252,"(0.022325537797245103, 0.047672488096919936)",200
1,AE,0.1718980073928833,"(0.11335342892791785, 0.23044258585784877)",200
2,CAE,0.16697466373443604,"(0.14552805985651707, 0.188421267612355)",200
3,VAE,0.16768383979797363,"(0.16129181575921256, 0.1740758638367347)",200
